# Exercícios PySpark — NYC Taxi Trip Data

Análise exploratória de ~4,1 milhões de corridas de táxi em Nova York (out/2024) usando PySpark.

## Sumário
1. [Questão 1 — Schema, amostra e contagem](#Questão-1)
2. [Questão 2 — Seleção de colunas](#Questão-2)
3. [Questão 3 — Filtro combinado](#Questão-3)
4. [Questão 4 — inferSchema vs StructType (dissertativa)](#Questão-4)
5. [Questão 5 — Receita por forma de pagamento](#Questão-5)
6. [Questão 6 — Tarifa e distância média por hora](#Questão-6)
7. [Questão 7 — Transformações vs ações (dissertativa)](#Questão-7)
8. [Questão 8 — Percentual de gorjeta](#Questão-8)
9. [Questão 9 — Join com tabela de zonas](#Questão-9)
10. [Questão 10 — count() vs groupBy() e shuffle (dissertativa)](#Questão-10)


# Preparação do ambiente

Download do dataset, criação da SparkSession e imports usados no notebook inteiro

In [20]:
from pyspark.sql import functions as F
import time

!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv
from pyspark.sql import SparkSession
spark = (
SparkSession.builder
.appName("ExerciciosPySpark")
.master("local[*]")
.getOrCreate()
)
df = spark.read.csv("nyc_tripdata_2024_sample_4M.csv", header=True,
inferSchema=True)

## Questão 1

Depois de carregar o DataFrame df, exiba:

a) o schema inferido (tipos de cada coluna);

b) as 10 primeiras linhas;

c) o número total de linhas do dataset.


In [21]:
# a) Exibir o schema inferido
df.printSchema()

# b) Exibir as 10 primeiras linhas
df.show(10)

# c) Exibir o número total de linhas
inicio = time.time()
total_linhas = df.count()
fim = time.time()

print("Número total de linhas:", total_linhas)
print("Tempo do count():", fim - inicio, "segundos")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+-----------

> **Conclusão:** o dataset tem **4.118.743 linhas** e 19 colunas, com tipos inferidos corretamente (datas como `timestamp`, valores monetários como `double`). O `count()` levou ~2,68s (valor usado como referência de comparação na questão 10).

## Questão 2

Selecione apenas as colunas VendorID, tpep_pickup_datetime, trip_distance, fare_amount e
payment_type, e exiba as 5 primeiras linhas do resultado.

In [22]:
# Seleciona apenas as colunas VendorID, tpep_pickup_datetime, trip_distance, fare_amount e payment_type, e exiba as 5 primeiras linhas do resultado.
df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
).show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


> **Conclusão:** seleção simples de colunas, sem shuffle envolvido.

## Questão 3
Filtre as corridas que atendem simultaneamente às duas condições abaixo, e exiba quantas corridas
restaram:
- trip_distance maior que 5 milhas;
- passenger_count maior ou igual a 3.

In [23]:
# Filtra as corridas que atendem simultaneamente às condições (trip_distance maior que 5 milhas; passenger_count maior ou igual a 3.), e exibe quantas corridas restaram.
df_filtrado = df.filter(
    (df.trip_distance > 5) &
    (df.passenger_count >= 3)
)
quantidade = df_filtrado.count()
print("Quantidade de corridas:", quantidade)

Quantidade de corridas: 50665


> **Conclusão:** apenas **50.665 corridas** (~1,2% do total) combinam trajeto longo (>5 milhas) com 3+ passageiros

## Questão 4

No comando de preparação deste exercício, o DataFrame foi carregado com inferSchema=True, deixando o
Spark descobrir sozinho o tipo de cada coluna. Explique como esse processo de inferência funciona, e
compare com a alternativa de definir o schema manualmente usando StructType/StructField (como
fizemos no notebook do Colab da aula anterior). Quais são as vantagens e os riscos de cada abordagem,
especialmente pensando num arquivo com milhões de linhas como o deste exercício?

Quando usamos `inferSchema=True`, o Spark analisa os dados do arquivo para tentar identificar automaticamente o tipo de cada coluna. Assim, valores podem ser reconhecidos como `IntegerType`, `DoubleType`, `TimestampType`, `StringType`, entre outros. Essa abordagem facilita o carregamento porque não precisamos informar previamente a estrutura dos dados.

A principal vantagem do `inferSchema=True` é a praticidade, principalmente quando ainda estamos explorando uma base ou não conhecemos completamente sua estrutura. Porém, a inferência exige que o Spark examine os dados para determinar os tipos, o que aumenta o custo de leitura. Em arquivos muito grandes, como uma base com milhões de linhas, isso pode aumentar o tempo de processamento. Além disso, existe o risco de o tipo inferido não ser o esperado caso os dados apresentem inconsistências.

Na definição manual, utilizamos `StructType` e `StructField` para informar explicitamente o nome e o tipo de cada coluna. Nesse caso, o Spark não precisa descobrir os tipos, pois eles já foram definidos pelo programador. Isso torna a leitura mais previsível e pode ser mais eficiente em grandes volumes de dados. Por outro lado, exige conhecer previamente a estrutura do arquivo e definir corretamente cada coluna. Se o schema informado não corresponder aos dados reais, podem ocorrer valores nulos inesperados ou problemas durante o processamento.

Portanto, `inferSchema=True` é conveniente para exploração e desenvolvimento inicial, enquanto a definição manual do schema é geralmente mais adequada quando a estrutura dos dados já é conhecida, especialmente em arquivos grandes e processos que serão executados repetidamente, pois oferece maior controle, consistência e eficiência.


## Questão 5
Agrupe as corridas por payment_type e calcule, para cada grupo:
- a quantidade de corridas;
- a soma total de total_amount (receita total).

Exiba o resultado ordenado pela receita total, da maior para a menor.

In [24]:
inicio = time.time()
resultado = (
    df.groupBy("payment_type").agg(F.count("*").alias("quantidade_corridas"),F.sum("total_amount").alias("receita_total"))
      .orderBy(F.desc("receita_total"))
)

resultado.show()

fim = time.time()
print("Tempo do groupBy():", fim - inicio, "segundos")

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|       receita_total|
+------------+-------------------+--------------------+
|           1|            3045849| 9.116799616010016E7|
|           2|             553536|1.2987084559999354E7|
|           0|             410746|1.0123049400000528E7|
|           3|              29100|  220775.24999999974|
|           4|              79511|  133192.01999999984|
|           5|                  1|                62.0|
+------------+-------------------+--------------------+

Tempo do groupBy(): 9.142404794692993 segundos


> **Conclusão:** pagamento em **cartão de crédito (`payment_type=1`)** domina a receita (R\$ 91,2 milhões), quase 7x mais que dinheiro (`payment_type=2`, R\$ 13,0 milhões).


## Questão 6
Crie uma nova coluna chamada hora_embarque, extraindo apenas a hora (0 a 23) da coluna
tpep_pickup_datetime. Em seguida, agrupe por hora_embarque e calcule a tarifa média (fare_amount) e a
distância média (trip_distance) para cada hora do dia. Exiba o resultado ordenado pela hora, de 0 a 23.

In [25]:
# Cria a coluna com a hora do embarque
df_hora = df.withColumn(
    "hora_embarque",F.hour("tpep_pickup_datetime")
)

# Agrupa por hora e calcular as médias
resultado = (
    df_hora.groupBy("hora_embarque").agg(
        F.avg("fare_amount").alias("tarifa_media"),
        F.avg("trip_distance").alias("distancia_media")
    )
    .orderBy("hora_embarque")
)
resultado.show(24)

+-------------+------------------+------------------+
|hora_embarque|      tarifa_media|   distancia_media|
+-------------+------------------+------------------+
|            0| 19.72867660335703| 5.130178643081671|
|            1| 17.54847894641617| 3.739999483030465|
|            2|16.426538461538446| 4.542445678033303|
|            3| 17.24064640950263| 3.401675265462839|
|            4| 22.33585451861572|11.412191651631977|
|            5|26.226564065583663| 23.33988120540463|
|            6|21.931859821807333|14.540589393296582|
|            7| 19.33026953083623|11.087329050022918|
|            8|18.511148615351107| 8.533842520592145|
|            9|18.400291333656767| 5.604490502277248|
|           10| 18.56454835768904|4.5114450807098905|
|           11|18.851552824117647| 4.075729557436569|
|           12|19.207714073999153| 4.468694683646846|
|           13| 19.95819012628161| 5.262006204074442|
|           14|20.604332332373676| 4.622973056355365|
|           15| 20.757107834

> **Conclusão:** A **distância média** tem um pico bem mais marcante às 5h-6h da manhã

## Questão 7

No Spark, existe uma distinção entre transformações (transformations) e ações (actions). Usando como
exemplo os comandos que você utilizou nas Questões 2, 3 e 5, explique essa diferença. Por que se diz que o
Spark utiliza avaliação preguiçosa (lazy evaluation), e qual é a vantagem prática disso?

No Spark, as operações podem ser divididas em **transformações (transformations)** e **ações (actions)**.

As **transformações** criam um novo DataFrame a partir de outro, mas não executam imediatamente o processamento dos dados. Por exemplo, na Questão 2, o `select()` foi utilizado para selecionar determinadas colunas. Na Questão 3, o `filter()` foi utilizado para filtrar as corridas de acordo com algumas condições. Já na Questão 5, foram utilizados groupBy(), agg() e orderBy() para agrupar as corridas por payment_type, calcular a quantidade de corridas e a soma de total_amount, e ordenar o resultado pela receita total. Essas operações são transformações.

As **ações**, por outro lado, fazem com que o Spark execute de fato as transformações que estavam pendentes e produza um resultado. Nos exercícios, exemplos de ações são `show()`, utilizado para exibir os dados, e `count()`, utilizado para retornar a quantidade de linhas.

O Spark utiliza **avaliação preguiçosa (lazy evaluation)** porque, ao receber uma sequência de transformações, ele não executa cada uma imediatamente. Em vez disso, registra as operações que deverão ser realizadas. A execução acontece somente quando uma ação, como `show()` ou `count()`, é chamada.

A principal vantagem dessa estratégia é que o Spark consegue analisar o conjunto de transformações antes da execução e otimizar o plano de processamento. Por exemplo, ele pode evitar operações desnecessárias e organizar a execução de maneira mais eficiente. Isso é especialmente importante ao trabalhar com milhões de registros, pois reduz processamento desnecessário e pode melhorar o desempenho da aplicação.


## Questão 8
 Considerando apenas as corridas em que total_amount seja maior que zero, crie uma nova coluna chamada percentual_gorjeta, calculada como (tip_amount / total_amount) * 100. Em seguida, exiba as 10 corridas com maior percentual_gorjeta, mostrando as colunas VendorID, total_amount, tip_amount e percentual_gorjeta.

In [26]:
resultado = (
  df.filter(F.col("total_amount") > 0)
    .withColumn("percentual_gorjeta", (F.col("tip_amount") / F.col("total_amount")) * 100)
      .select(
          "VendorID",
          "total_amount",
          "tip_amount",
          "percentual_gorjeta")
      .orderBy(F.desc("percentual_gorjeta"))
)
resultado.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27| 323.3128834355828|
|       2|        2.07|      3.68| 177.7777777777778|
|       2|         1.6|      2.82|176.24999999999997|
|       2|        2.33|      3.72|159.65665236051504|
|       2|        2.54|      3.76|148.03149606299212|
|       2|        3.76|      3.96|105.31914893617022|
|       2|        39.7|      40.0|100.75566750629723|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0| 99.49238578680203|
|       1|       150.0|     149.0| 99.33333333333333|
+--------+------------+----------+------------------+
only showing top 10 rows


> **Conclusão:** os maiores percentuais de gorjeta (>100%) aparecem em corridas com `total_amount` muito baixo, provável inconsistência nos dados brutos, não um padrão real de comportamento de gorjeta.

## Questão 9

A NYC TLC disponibiliza uma tabela de referência que traduz os códigos de zona usados em
PULocationID/DOLocationID para o nome do bairro (Borough) e da zona (Zone) correspondente. Baixe essa
tabela e carregue em um novo DataFrame e em seguida:

a) Faça o join entre df e zonas, relacionando df.PULocationID com zonas.LocationID, para descobrir o
bairro (Borough) de onde cada corrida partiu;

b) Agrupe o resultado por Borough e conte quantas corridas tiveram origem em cada um;

c) Exiba o resultado ordenado do bairro com mais corridas para o com menos.



In [27]:
# tabela de referência

!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv
zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
zonas.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [28]:
from pyspark.sql import functions as F

# a) Join entre as corridas e a tabela de zonas
df_join = df.join(zonas, df.PULocationID == zonas.LocationID,"inner")

# b) Agrupar por Borough e contar as corridas
resultado = (df_join.groupBy("Borough").agg(
        F.count("*").alias("quantidade_corridas"))

    # c) Ordenar do bairro com mais corridas para o com menos
    .orderBy(F.desc("quantidade_corridas"))
)

resultado.show()

+-------------+-------------------+
|      Borough|quantidade_corridas|
+-------------+-------------------+
|    Manhattan|            3641752|
|       Queens|             388736|
|     Brooklyn|              60200|
|        Bronx|              12702|
|      Unknown|              12172|
|          N/A|               2421|
|          EWR|                565|
|Staten Island|                195|
+-------------+-------------------+



> **Conclusão:** **Manhattan** concentra ≈ 88% das corridas (3.641.752 de 4.118.743), seguido de longe por Queens (≈ 9%). A soma das corridas por bairro bate exatamente com o total da Questão 1, confirmando que o `inner join` não descartou nenhuma linha.

## Questão 10
Compare o tempo de execução do count() da Questão 1 (sem nenhum agrupamento) com o tempo de
execução do groupBy() da Questão 5. Baseando-se no conceito de shuffle, explique por que operações de
agrupamento tendem a ser mais custosas do que operações de filtragem ou seleção de colunas, mesmo
processando o mesmo volume de dados.


Na execução realizada, o `count()` da Questão 1 levou aproximadamente **2,68 segundos**, enquanto a operação com `groupBy()` da Questão 5 levou aproximadamente **9,14 segundos**. Portanto, a operação de agrupamento levou cerca de **4,14 vezes mais tempo**.

Essa diferença ocorre principalmente por causa do **shuffle**. No `count()`, o Spark percorre os dados para contabilizar os registros, sem precisar agrupá-los de acordo com uma chave.

Já no `groupBy()` da Questão 5, os registros com o mesmo `payment_type` precisam ser reunidos para calcular a quantidade de corridas e a soma de `total_amount`. Como os registros podem estar distribuídos em diferentes partições, o Spark precisa redistribuir os dados para colocar registros com a mesma chave juntos. Essa movimentação entre partições é chamada de **shuffle**.

O shuffle é uma operação mais custosa porque envolve reorganização e transferência de dados entre partições e, em um cluster, pode envolver comunicação entre diferentes máquinas, além do uso de memória e disco.

Operações como `filter()` e `select()` geralmente não exigem essa redistribuição. Elas podem ser executadas localmente em cada partição, verificando registros ou selecionando colunas. Por isso, tendem a ser menos custosas.

Assim, os tempos observados são coerentes com esse comportamento: o `groupBy()` apresentou um tempo de execução maior porque exige agrupamento, agregação, ordenação e movimentação de dados por meio de shuffle.
